In [1]:
# https://www.kaggle.com/competitions/latent-model-classification-aicc-round-0/

import os
from pathlib import Path
import random
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy.linalg as la
from sklearn.metrics import accuracy_score, confusion_matrix
from tqdm.auto import tqdm

SEED = 12345
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

INPUT_DIM = 100
EMBED_DIM = 10
LOGIT_DIM = 5
N = 5000
P_A = 0.5

In [2]:
OUT_DIR = Path("/kaggle/input/latent-model-classification-aicc-round-0/")
TRAIN_CSV = OUT_DIR / "dataset.csv"
PEN_A_PTH = OUT_DIR / "modelA_penultimate.pth"
PEN_B_PTH = OUT_DIR / "modelB_penultimate.pth"

In [3]:
df = pd.read_csv(TRAIN_CSV)
x_cols = [c for c in df.columns if c.startswith("x")]
X = df[x_cols].values.astype(np.float32)
class PenultimateNet(nn.Module):
    def __init__(self, in_dim=INPUT_DIM, hidden=64, out_dim=EMBED_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_dim),
        )
    def forward(self, x):
        return self.net(x)

device = "cpu" # or "cuda"
penA = PenultimateNet().to(device)
penB = PenultimateNet().to(device)
penA.load_state_dict(torch.load(str(PEN_A_PTH), map_location=device))
penB.load_state_dict(torch.load(str(PEN_B_PTH), map_location=device))
penA.eval(); penB.eval()

PenultimateNet(
  (net): Sequential(
    (0): Linear(in_features=100, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [4]:
with torch.no_grad():
    Xt = torch.from_numpy(X).to(device)
    ZA = penA(Xt).cpu().numpy()
    ZB = penB(Xt).cpu().numpy()

Y = df[[f'y{i}' for i in range(5)]].values

ZA.shape, ZB.shape, Y.shape

((5000, 10), (5000, 10), (5000, 5))

In [5]:
A_indices = np.arange(5000)
B_indices = np.array([], dtype='int64')

epochs = 2500
no_improve_epochs = 100
log_rate = 25

mean_wait_time = 0
for epoch in (pbar := tqdm(range(epochs), desc='Epoch')):
    w, res, *_ = np.linalg.lstsq(ZA[A_indices], Y[A_indices])
    best_score, best_leave = 2*res.mean(), -1
    cur_wait, total_epoch_wait = 0, 0
    for i, el in enumerate(A_indices):
        total_epoch_wait += 1
        _, reselA, *_ = np.linalg.lstsq(ZA[np.delete(A_indices, i)], Y[np.delete(A_indices, i)])
        _, reselB, *_ = np.linalg.lstsq(ZB[np.append(B_indices, el)], Y[np.append(B_indices, el)])
        if reselB.size == 0:
            score = 2*reselA.mean()
        else:
            score = reselA.mean() + reselB.mean()
        if score < best_score:
            best_score, best_leave = score, i
            cur_wait = 0
        else:
            cur_wait += 1
            if cur_wait >= no_improve_epochs:
                break
                
    mean_wait_time += total_epoch_wait

    pbar.set_postfix({'mean_wait_time': f'{mean_wait_time/(epoch+1):.2f}'})
    
    if (epoch+1) % log_rate == 0:
        
        print(f'Epoch: {epoch+1}/{epochs} | Score: {best_score:.1f}')
    
    if best_leave == -1:
        print(f'Stopping early at epoch {epoch+1}/{epochs}')
        break

    B_indices = np.append(B_indices, A_indices[best_leave])
    A_indices = np.delete(A_indices, best_leave)

np.save('A_indices.npy', A_indices)
np.save('B_indices.npy', B_indices)

Epoch:   0%|          | 0/2500 [00:00<?, ?it/s]

Epoch: 25/2500 | Score: 1135.7
Epoch: 50/2500 | Score: 1125.5
Epoch: 75/2500 | Score: 1111.5
Epoch: 100/2500 | Score: 1101.0
Epoch: 125/2500 | Score: 1088.2
Epoch: 150/2500 | Score: 1079.9
Epoch: 175/2500 | Score: 1071.3
Epoch: 200/2500 | Score: 1056.1
Epoch: 225/2500 | Score: 1048.1
Epoch: 250/2500 | Score: 1041.0
Epoch: 275/2500 | Score: 1035.5
Epoch: 300/2500 | Score: 1030.3
Epoch: 325/2500 | Score: 1018.4
Epoch: 350/2500 | Score: 1006.5
Epoch: 375/2500 | Score: 997.9
Epoch: 400/2500 | Score: 991.7
Epoch: 425/2500 | Score: 983.1
Epoch: 450/2500 | Score: 970.5
Epoch: 475/2500 | Score: 964.3
Epoch: 500/2500 | Score: 953.7
Epoch: 525/2500 | Score: 943.9
Epoch: 550/2500 | Score: 939.5
Epoch: 575/2500 | Score: 936.1
Epoch: 600/2500 | Score: 932.1
Epoch: 625/2500 | Score: 928.5
Epoch: 650/2500 | Score: 922.5
Epoch: 675/2500 | Score: 907.8
Epoch: 700/2500 | Score: 897.8
Epoch: 725/2500 | Score: 890.4
Epoch: 750/2500 | Score: 877.4
Epoch: 775/2500 | Score: 870.9
Epoch: 800/2500 | Score: 866

In [6]:
labels = np.array(['B'] * 5000)
labels[A_indices] = 'A'

subm = pd.DataFrame({"ID": np.arange(1, 5000+1), "source": labels})

subm.to_csv("submission.csv", index=False)
subm.head()

,ID,source
0,1,B
1,2,B
2,3,B
3,4,B
4,5,A
